# Loan Prediction System with Streamlit GUI

## Objective
The objective of this project is to build a machine learning system that predicts whether a loan application will be approved or rejected.

## Project Workflow
1. Import libraries
2. Load dataset
3. Data preprocessing & handling missing values
4. Encoding categorical variables
5. Feature scaling
6. Training Logistic Regression, Random Forest, XGBoost
7. Model evaluation & saving
8.  Interactive Streamlit GUI

# Step 1:Import libraries

In [19]:
# ============================================================
# Import required libraries
# ============================================================

# Data manipulation
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Machine learning models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Data preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
# Model evaluation
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Save model
import joblib
print("✅ Libraries Imported Successfully")
# -------------------------------
# Ignore Warnings
# -------------------------------
import warnings

# Ignore all warnings
warnings.filterwarnings('ignore')

print("✅ All warnings will be suppressed")

✅ Libraries Imported Successfully
✅ All warnings will be suppressed


# Step 2: Load Dataset

In [10]:
# =========================================================
#  Load Dataset
# =========================================================

# Load train dataset
train_df = pd.read_csv("train_loan.csv")

# Show first 5 rows
print("📌 First 5 rows of dataset:")
display(train_df.head())

# Check missing values
print("\n📌 Missing values in each column:")
print(train_df.isnull().sum())

📌 First 5 rows of dataset:


,Loan_ID,Gender,Married,Dependents,Education,Self_Employed,ApplicantIncome,CoapplicantIncome,LoanAmount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,LP001002,Male,No,0,Graduate,No,5849,0.0,NaN,360.0,1.0,Urban,Y
1,LP001003,Male,Yes,1,Graduate,No,4583,1508.0,128.0,360.0,1.0,Rural,N
2,LP001005,Male,Yes,0,Graduate,Yes,3000,0.0,66.0,360.0,1.0,Urban,Y
3,LP001006,Male,Yes,0,Not Graduate,No,2583,2358.0,120.0,360.0,1.0,Urban,Y
4,LP001008,Male,No,0,Graduate,No,6000,0.0,141.0,360.0,1.0,Urban,Y



📌 Missing values in each column:
Loan_ID               0
Gender               13
Married               3
Dependents           15
Education             0
Self_Employed        32
ApplicantIncome       0
CoapplicantIncome     0
LoanAmount           22
Loan_Amount_Term     14
Credit_History       50
Property_Area         0
Loan_Status           0
dtype: int64


# Step 3: Handle Missing Values

In [11]:
# =========================================================
# Handle Missing Values
# =========================================================

# Fill categorical columns with mode
categorical_cols = ["Gender", "Married", "Dependents", "Self_Employed", "Credit_History"]
for col in categorical_cols:
    train_df[col] = train_df[col].fillna(train_df[col].mode()[0])
    print(f"✅ Filled missing values in {col} with mode: {train_df[col].mode()[0]}")

# Fill numerical columns with median
numerical_cols = ["LoanAmount", "Loan_Amount_Term"]
for col in numerical_cols:
    train_df[col] = train_df[col].fillna(train_df[col].median())
    print(f"✅ Filled missing values in {col} with median: {train_df[col].median()}")

# Verify missing values are handled
print("\n📌 Missing values after handling:")
print(train_df.isnull().sum())

✅ Filled missing values in Gender with mode: Male
✅ Filled missing values in Married with mode: Yes
✅ Filled missing values in Dependents with mode: 0
✅ Filled missing values in Self_Employed with mode: No
✅ Filled missing values in Credit_History with mode: 1.0
✅ Filled missing values in LoanAmount with median: 128.0
✅ Filled missing values in Loan_Amount_Term with median: 360.0

📌 Missing values after handling:
Loan_ID              0
Gender               0
Married              0
Dependents           0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64


# Step 4: Encode Categorical Variables

In [12]:
# =========================================================
#  Encode Categorical Variables
# =========================================================

# Initialize dictionary to store label encoders
le_dict = {}

# Encode categorical columns
categorical_cols = ["Gender", "Married", "Dependents", "Education", "Self_Employed", "Property_Area"]
for col in categorical_cols:
    le = LabelEncoder()
    train_df[col] = le.fit_transform(train_df[col])
    le_dict[col] = le
    print(f"✅ Encoded column '{col}' with classes: {le.classes_}")

# Encode target
target_le = LabelEncoder()
train_df['Loan_Status'] = target_le.fit_transform(train_df['Loan_Status'])
print(f"✅ Encoded target 'Loan_Status' with classes: {target_le.classes_}")

✅ Encoded column 'Gender' with classes: ['Female' 'Male']
✅ Encoded column 'Married' with classes: ['No' 'Yes']
✅ Encoded column 'Dependents' with classes: ['0' '1' '2' '3+']
✅ Encoded column 'Education' with classes: ['Graduate' 'Not Graduate']
✅ Encoded column 'Self_Employed' with classes: ['No' 'Yes']
✅ Encoded column 'Property_Area' with classes: ['Rural' 'Semiurban' 'Urban']
✅ Encoded target 'Loan_Status' with classes: ['N' 'Y']


# Step 5: Feature Selection & Train/Test Split

In [13]:
# =========================================================
#  Feature Selection & Train/Test Split
# =========================================================

# Separate features and target
X = train_df.drop(columns=["Loan_ID", "Loan_Status"])
y = train_df["Loan_Status"]

# Split into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"✅ Training set: {X_train.shape}, Validation set: {X_val.shape}")

✅ Training set: (491, 11), Validation set: (123, 11)


# Step 6: Feature Scaling

In [17]:
# =========================================================
#  Feature Scaling
# =========================================================

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("✅ Features scaled using StandardScaler")

✅ Features scaled using StandardScaler


# Step 7: Train Machine Learning Models

In [20]:
# =========================================================
#  Train Machine Learning Models
# =========================================================

# Initialize models
models = {
    "Logistic Regression": LogisticRegression(max_iter=500),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# Train and evaluate models
results = {}
for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
    
    acc = accuracy_score(y_val, y_pred)
    results[name] = acc
    print(f"✅ {name} Accuracy: {acc:.4f}")

✅ Logistic Regression Accuracy: 0.7886
✅ Random Forest Accuracy: 0.7561
✅ XGBoost Accuracy: 0.7398


# Step 8: Save the Best Model

In [21]:
# =========================================================
#  Save the Best Model
# =========================================================

# Find best model
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

# Save model and scaler
joblib.dump(best_model, "best_model.pkl")
joblib.dump(scaler, "scaler.pkl")
print(f"✅ Best model saved: {best_model_name}")

✅ Best model saved: Logistic Regression


# Conclusion

In this project, we developed a **Machine Learning system to predict loan approval status** based on applicant information. The goal was to analyze financial and demographic features and determine whether a loan application is likely to be approved.

## Project Workflow

The following steps were performed during the project:

### 1. Data Loading 
- Loaded the loan dataset using **Pandas**.

### 2. Data Preprocessing
- Handled **missing values** using appropriate techniques such as filling with mean or mode.
- Encoded **categorical variables** into numerical form using label encoding.
- Prepared the dataset for machine learning models.

### 3. Feature Scaling
- Applied **StandardScaler** to normalize numerical features.
- Saved the scaler for use during prediction.

### 4. Model Training
We trained and compared multiple machine learning models:
- **Logistic Regression**
- **Random Forest**
- **XGBoost**

Each model was trained using the processed dataset.

### 5. Model Evaluation
- Evaluated model performance using **accuracy score**.
- Compared results to determine the best-performing model.

### 6. Model Selection
- The model with the highest accuracy was selected as the **best model**.
- The trained model was saved using **Joblib** for future predictions.

### 7. Model Deployment
- Built a **Streamlit web application** to allow users to input loan applicant information.
- The application uses the trained model to predict **loan approval status in real time**.

## Final Outcome
This project demonstrates a complete **end-to-end machine learning workflow**, including:
- Data preprocessing
- Model training and evaluation
- Model saving
- Web application deployment using Streamlit

The system can now be used to assist in predicting whether a loan application is likely to be **approved or rejected** based on the provided inputs.